# Entrenamiento en Google Colab

Entorno → Cambiar tipo de entorno de ejecución → **GPU T4**.

Flujo:
1. Montar Drive y clonar el repositorio.
2. Copiar los rostros ya extraídos (`.zip` en Drive) al disco local de Colab — leer miles de imágenes directo desde Drive es muy lento.
3. Entrenar guardando checkpoints en Drive. Si la sesión se cae, se vuelve a correr todo y la celda de entrenamiento continúa con `--reanudar`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Carpeta del proyecto en Drive (ajustar si es necesario)
DRIVE = '/content/drive/MyDrive/deepfakes'
!mkdir -p {DRIVE}/runs {DRIVE}/results

## Código

Si el repositorio es privado, crea un *token* de GitHub (solo lectura) y guárdalo en los **Secretos** de Colab (icono de llave) con el nombre `GITHUB_TOKEN`.

In [ ]:
import os
REPO = 'Julianchu850/proyecto-redes-neuronales-deteccion-deepfake'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    url = f'https://{token}@github.com/{REPO}.git'
except Exception:
    url = f'https://github.com/{REPO}.git'

if not os.path.exists('/content/repo'):
    !git clone -q {url} /content/repo
else:
    !git -C /content/repo pull -q
%cd /content/repo
!pip install -q -r requirements.txt

## Datos

Se espera en Drive `deepfakes/faces_ffpp.zip`, generado por `00_preparar_datos.ipynb` (o en el Mac con `src.data.extraer_rostros`).

In [ ]:
!mkdir -p /content/data/faces
!unzip -q -n {DRIVE}/faces_ffpp.zip -d /content/data/faces/
!ls /content/data/faces

In [ ]:
import os
# Splits: los del repositorio, o los que guardó 00_preparar_datos en Drive, o se crean aquí
if not os.path.exists('data/splits/ffpp/train.csv'):
    if os.path.exists(f'{DRIVE}/splits/ffpp/train.csv'):
        !mkdir -p data/splits && cp -r {DRIVE}/splits/ffpp data/splits/
    else:
        !python -m src.data.crear_splits --dataset ffpp --rostros /content/data/faces/ffpp --salida data/splits/ffpp
!ls data/splits/ffpp

## Entrenamiento — línea base MesoNet (semana 1)

In [ ]:
MODELO = 'meso4'
RUN = f'{DRIVE}/runs/{MODELO}'
!python -m src.entrenar --modelo {MODELO} --datos /content/data/faces/ffpp --splits data/splits/ffpp \
    --salida {RUN} --epocas 30 --batch 64 --workers 2 --reanudar

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
h = pd.read_csv(f'{RUN}/historial.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h.epoca, h.perdida_train); ax[0].set_title('Pérdida de entrenamiento'); ax[0].set_xlabel('época')
ax[1].plot(h.epoca, h.auc_val_frame, label='frame'); ax[1].plot(h.epoca, h.auc_val_video, label='video')
ax[1].set_title('AUC de validación'); ax[1].set_xlabel('época'); ax[1].legend()
plt.tight_layout()

## Evaluación en el conjunto de prueba (E1)

In [ ]:
!python -m src.evaluar --checkpoint {RUN}/mejor.pt --datos /content/data/faces/ffpp \
    --csv data/splits/ffpp/test.csv --salida {DRIVE}/results/{MODELO}_ffpp_test

## Semana 2 — Red propia, ablación (E1, E2) y referencia preentrenada

Cada experimento se entrena y luego se evalúa en el test de FF++. Como los checkpoints van a Drive con `--reanudar`, si Colab se cae basta con volver a correr esta celda: los experimentos terminados se saltan y el que iba a medias continúa.

Sugerencia: correr algunos en la GTX 1660 en paralelo (mismos comandos, cambiando las rutas).

In [ ]:
import os
EXPERIMENTOS = {
    # nombre del run: (modelo, argumentos extra)
    'dos_ramas':        ('dos_ramas', ''),
    'solo_espacial':    ('solo_espacial', ''),
    'solo_frecuencia':  ('solo_frecuencia', ''),
    'dos_ramas_sin_se': ('dos_ramas_sin_se', ''),
    'dos_ramas_srm':    ('dos_ramas_srm', ''),
    'efficientnet_b0':  ('efficientnet_b0', '--lr 3e-4 --epocas 15'),
}

for nombre, (modelo, extra) in EXPERIMENTOS.items():
    run = f'{DRIVE}/runs/{nombre}'
    res = f'{DRIVE}/results/{nombre}_ffpp_test'
    if os.path.exists(f'{res}/metricas.json'):
        print(f'✔ {nombre} ya terminado'); continue
    print(f'▶ {nombre}')
    !python -m src.entrenar --modelo {modelo} --datos /content/data/faces/ffpp --splits data/splits/ffpp \
        --salida {run} --workers 2 --reanudar {extra}
    if not os.path.exists(f'{run}/TERMINADO'):
        print(f'✗ {nombre} no terminó (error o pausa): vuelve a correr la celda para continuar'); break
    !python -m src.evaluar --checkpoint {run}/mejor.pt --datos /content/data/faces/ffpp \
        --csv data/splits/ffpp/test.csv --salida {res}

## E3 — Técnica de manipulación no vista (*leave-one-out*)

Se entrena con tres técnicas y se evalúa el AUC sobre la cuarta (ver `por_metodo` en las métricas).

In [ ]:
TECNICAS = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']
for modelo in ['meso4', 'dos_ramas', 'solo_espacial']:
    for fuera in TECNICAS:
        nombre = f'{modelo}_sin_{fuera}'
        run, res = f'{DRIVE}/runs/e3/{nombre}', f'{DRIVE}/results/e3/{nombre}'
        if os.path.exists(f'{res}/metricas.json'):
            print(f'✔ {nombre} ya terminado'); continue
        entrenar_con = ' '.join(t for t in TECNICAS if t != fuera)
        !python -m src.entrenar --modelo {modelo} --datos /content/data/faces/ffpp --splits data/splits/ffpp \
            --salida {run} --workers 2 --reanudar --metodos {entrenar_con}
        if not os.path.exists(f'{run}/TERMINADO'):
        print(f'✗ {nombre} no terminó (error o pausa): vuelve a correr la celda para continuar'); break
    !python -m src.evaluar --checkpoint {run}/mejor.pt --datos /content/data/faces/ffpp \
            --csv data/splits/ffpp/test.csv --salida {res}

## E3b — FaceShifter: técnica que ningún modelo vio

Si el dataset trae FaceShifter, se extrajo pero nunca se usó para entrenar. Es la prueba más limpia de generalización dentro de FF++.

In [ ]:
for nombre in ['meso4', 'dos_ramas', 'solo_espacial', 'solo_frecuencia', 'efficientnet_b0']:
    !python -m src.evaluar --checkpoint {DRIVE}/runs/{nombre}/mejor.pt --datos /content/data/faces/ffpp \
        --csv data/splits/ffpp/test.csv --salida {DRIVE}/results/{nombre}_faceshifter --metodos FaceShifter

## E4 — Generalización a Celeb-DF v2

Requiere los rostros de Celeb-DF en Drive como `faces_celebdf.zip` (se generan igual que los de FF++, con `--dataset celebdf`) y el archivo `List_of_testing_videos.txt` del dataset.

In [ ]:
if os.path.exists(f'{DRIVE}/faces_celebdf.zip'):
    !unzip -q -n {DRIVE}/faces_celebdf.zip -d /content/data/faces/
    !python -m src.data.crear_splits --dataset celebdf --rostros /content/data/faces/celebdf \
        --lista-celebdf {DRIVE}/List_of_testing_videos.txt --salida data/splits/celebdf
    for nombre in ['meso4', 'dos_ramas', 'solo_espacial', 'solo_frecuencia', 'efficientnet_b0']:
        !python -m src.evaluar --checkpoint {DRIVE}/runs/{nombre}/mejor.pt --datos /content/data/faces/celebdf \
            --csv data/splits/celebdf/test.csv --salida {DRIVE}/results/{nombre}_celebdf
else:
    print('Falta faces_celebdf.zip en Drive')

## E5 — Grad-CAM

In [ ]:
from IPython.display import Image as Imagen, display
for nombre in ['meso4', 'dos_ramas', 'efficientnet_b0']:
    sal = f'{DRIVE}/results/gradcam/{nombre}'
    !python -m src.gradcam --checkpoint {DRIVE}/runs/{nombre}/mejor.pt --datos /content/data/faces/ffpp \
        --csv data/splits/ffpp/test.csv --salida {sal}
    display(Imagen(f'{sal}/gradcam_promedio.png'))

## Resumen de resultados

In [ ]:
import json, glob, pandas as pd
filas = []
for f in sorted(glob.glob(f'{DRIVE}/results/*/metricas.json')):
    m = json.load(open(f))
    fila = {'experimento': f.split('/')[-2], 'AUC frame': m['frame']['auc'], 'AUC video': m['video']['auc'],
            'acc video': m['video']['accuracy'], 'F1 video': m['video']['f1']}
    fila.update({f'AUC {k}': v['auc_video'] for k, v in m.get('por_metodo', {}).items()})
    filas.append(fila)
pd.DataFrame(filas).set_index('experimento').round(4)

## Demo

Crea un enlace público temporal para probar el modelo desde el navegador o el celular. Detén la celda para cerrarlo.

In [ ]:
!python -m app.demo --checkpoint {DRIVE}/runs/dos_ramas/mejor.pt {DRIVE}/runs/meso4/mejor.pt --compartir